In [3]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [8]:
print(train['Versuri'].apply(lambda x: len(x.split())).describe())

count    3415.000000
mean       19.867936
std         6.920662
min         3.000000
25%        15.000000
50%        18.000000
75%        23.000000
max        47.000000
Name: Versuri, dtype: float64


In [10]:
from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

train_text = train['Versuri'].str.lower()
test_text = test['Versuri'].str.lower()

le = LabelEncoder()
labels = le.fit_transform(train['Autor'])

clf1 = LogisticRegression(max_iter=2000)
clf2 = SGDClassifier(loss='modified_huber', penalty='elasticnet')
clf3 = MultinomialNB()

ensemble = VotingClassifier(
    estimators=[('lr', clf1), ('sgd', clf2), ('nb', clf3)],
    voting="soft"
)

# Pipeline-ul principal
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True)),
    ('ensemble', ensemble)
])

param_dist = {
    # Strategy for short texts, analyze both words and characters
    'tfidf__analyzer': ['word', 'char_wb'], 
    'tfidf__ngram_range': [(1, 2), (1, 3), (2, 4)],
    'tfidf__max_features': [10000, 20000, 50000],
    'tfidf__sublinear_tf': [True],
    
    # Logistic Regression
    'ensemble__lr__C': [0.1, 1, 10, 100],
    'ensemble__lr__solver': ['lbfgs', 'saga'],
    
    # SGD
    'ensemble__sgd__alpha': [1e-5, 1e-4, 1e-3],
    
    # Naive Bayes
    'ensemble__nb__alpha': [0.01, 0.01, 0.1, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rs = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=cv,
    scoring='accuracy',
    verbose=2,
    n_jobs=-1,
    random_state=42
)

rs.fit(train_text, labels)
model = rs.best_estimator_
predictions = model.predict(test_text)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END ensemble__lr__C=100, ensemble__lr__solver=lbfgs, ensemble__nb__alpha=1.0, ensemble__sgd__alpha=1e-05, tfidf__analyzer=word, tfidf__max_features=10000, tfidf__ngram_range=(1, 3), tfidf__sublinear_tf=True; total time=   1.7s
[CV] END ensemble__lr__C=100, ensemble__lr__solver=lbfgs, ensemble__nb__alpha=1.0, ensemble__sgd__alpha=1e-05, tfidf__analyzer=word, tfidf__max_features=10000, tfidf__ngram_range=(1, 3), tfidf__sublinear_tf=True; total time=   1.7s
[CV] END ensemble__lr__C=100, ensemble__lr__solver=lbfgs, ensemble__nb__alpha=1.0, ensemble__sgd__alpha=1e-05, tfidf__analyzer=word, tfidf__max_features=10000, tfidf__ngram_range=(1, 3), tfidf__sublinear_tf=True; total time=   1.8s
[CV] END ensemble__lr__C=100, ensemble__lr__solver=lbfgs, ensemble__nb__alpha=1.0, ensemble__sgd__alpha=1e-05, tfidf__analyzer=word, tfidf__max_features=10000, tfidf__ngram_range=(1, 3), tfidf__sublinear_tf=True; total time=   1.9s
[CV] END en

In [11]:
rows = []
predictions = le.inverse_transform(predictions)
for id, pred in zip(test['Id'], predictions):
    rows.append({
        "Id": id,
        "Autor": pred
    })
sub = pd.DataFrame(rows)
sub.to_csv('submission.csv', index=False)